![](header.jpg)


# Unix Domain Sockets vs TCP

Kevin J. Walchko, Phd

---

This is inspired by [Myhro Blog](https://blog.myhro.info/2017/01/how-fast-are-unix-domain-sockets)

In [1]:
import multiprocessing as mp
import threading
import socket
import time
import os

# Server and Client

Communicating via TCP or UDS is very similar, so here are some functions that create a server and a client. Each takes arguments to determine if communications is via TCP or UDS.

- Their address family: `socket.AF_INET` (IP) and `socket.AF_UNIX` (Unix sockets).
- To bind a process using `socket.AF_UNIX`, the socket file should be removed and created again if it already exists.
- When using `socket.AF_INET`, the `socket.SO_REUSEADDR` flag have to be set in order to avoid `socket.error: [Errno 98] Address already in use` errors that may occur even when the socket is properly closed. This option tells the kernel to reuse the same port if there are connections in the `TIME_WAIT` state.

## TCP

- allows communications between processes running on different machines
- all messages have to pass through the TCP/IP stack, so this should be slower

## UDS

- since the communications is through a file handler, the processes have to be on the same machine
    - I wouldn't use a file associated with a network share, then you are doing some sort of tcp/uds thing

In [12]:
def server(e,AF,addr):
    
    duration = 5
    end = time.time() + duration

    try:
        sock = socket.socket(AF, socket.SOCK_STREAM)
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        sock.bind(addr)
        sock.listen(0)
    except Exception as ex:
        print(ex)
        e.clear()
    
    # print("Server ready")

    conn, _ = sock.accept()
    
    while (time.time() < end):
        conn.send(b'Hello there!')
        # time.sleep(0.5)

    conn.close()
    # print('*** Server is done ***')
    e.clear()

In [13]:
def client(e,AF,addr):
    msgs = 0

    # print('Receiving messages...')

    try:
        sock = socket.socket(AF, socket.SOCK_STREAM)
        sock.connect(addr)
    except Exception as ex:
        print(ex)
        e.clear()
    
    while e.is_set():
        data = sock.recv(32)
        msgs += 1
        print(f"Messages recieved: {msgs}", end="\r")
        
    sock.close()

    print(f'Total received {msgs} messages.')

# TCP

In [15]:
event = mp.Event()
event.set()

# host = socket.gethostbyname(socket.gethostname())
host = "0.0.0.0"
af = socket.AF_INET
addr = (host, 5555,)

print("host computer: {}".format(addr))

# print(af, addr)

s = threading.Thread(target=server, args=(event,af,addr,),name="server")
s.start()
print('Started {}'.format(s.name))

c = threading.Thread(target=client, args=(event,af,addr,), name="client")
c.start()
print('Started {}'.format(c.name))

s.join()
c.join()

for p in [s, c]:
    print('{} is alive: {}'.format(p.name, p.is_alive()))

host computer: ('0.0.0.0', 5555)
Started server
Started client
Total received 102669 messages.
server is alive: False
client is alive: False


# UDS

In [17]:
event.set()

af = socket.AF_UNIX
addr = '/tmp/uds_test'

# if the file already exists, you will get an address in use error
if os.path.exists(addr):
    os.remove(addr)

s = threading.Thread(target=server, args=(event,af,addr), name="server")
s.start()

print('Started {}'.format(s.name))

c = threading.Thread(target=client, args=(event,af,addr), name="client")
c.start()
print('Started {}'.format(c.name))

s.join()
c.join()

for p in [s, c]:
    print('{} is alive: {}'.format(p.name, p.is_alive()))

Started server
Started client
Total received 150996 messages.
server is alive: False
client is alive: False


# Results

These tests seem to be all over the place ... not sure if jupyter is interfering with the execution or not.

The original author listed above showed UDS consistantly being twice as fast as TCP.